In [1]:
import os

In [2]:
%pwd

'c:\\Users\\penze\\Desktop\\MLdev_ops\\student_Performance_p1\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\penze\\Desktop\\MLdev_ops\\student_Performance_p1'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [7]:
from src.student_Performance_p1.constants import *
from src.student_Performance_p1.utils.common import read_yaml, create_directories


In [33]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        # Access model_trainer section from config.yaml
        config = self.config.model_trainer

        # Create model trainer folder
        create_directories([config.root_dir])

        # Create model trainer config object
        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_path,
            alpha=self.params.ElasticNet.alpha,
            l1_ratio=self.params.ElasticNet.l1_ratio,
            target_column=self.schema.TARGET_COLUMN.name 
        )

        return model_trainer_config
    
    

In [34]:
import pandas as pd
import os 
from src.student_Performance_p1 import logger 
from sklearn.linear_model import ElasticNet
import joblib


In [44]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
    
    def train_model(self):
        logger.info("Loading training data...")
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

# the elastic can only handel number for now so we just use this 
        logger.info("Splitting data into features and target...")
        x_train = train_data.drop(columns=[self.config.target_column,"ocean_proximity"] ,axis = 1)
        y_train = train_data[self.config.target_column]
        x_test = test_data.drop(columns=[self.config.target_column,"ocean_proximity"],axis = 1)
        y_test = test_data[self.config.target_column]
        logger.info("Splitting data into features and target...")


        logger.info("Training ElasticNet model...")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        model.fit(x_train, y_train)

        logger.info("Saving trained model...")
        joblib.dump(model, self.config.model_name)

In [45]:
try: 
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    logger.exception(e)

[2026-05-27 10:58:18,289: INFO: common: YAML file 'config\config.yaml' Loading successfully.]
[2026-05-27 10:58:18,290: INFO: common: YAML file 'params.yaml' Loading successfully.]
[2026-05-27 10:58:18,292: INFO: common: YAML file 'schema.yaml' Loading successfully.]
[2026-05-27 10:58:18,292: INFO: common: Directories created successfully: ['artifacts']]
[2026-05-27 10:58:18,293: INFO: common: Directories created successfully: ['artifacts/model_trainer']]
[2026-05-27 10:58:18,294: INFO: 3085663722: Loading training data...]
[2026-05-27 10:58:18,308: INFO: 3085663722: Splitting data into features and target...]
[2026-05-27 10:58:18,310: INFO: 3085663722: Splitting data into features and target...]
[2026-05-27 10:58:18,310: INFO: 3085663722: Training ElasticNet model...]
[2026-05-27 10:58:19,113: INFO: 3085663722: Saving trained model...]
